# Week 3 Exercise — Stayez Synthetic Recommendation Data Generator

**Student:** Vagz1216  
**Course:** LLM Engineering — Andela AI Engineering Bootcamp  
**Exercise:** Week 3 — Synthetic Data Generator

---

## Project Overview
For Week 3, I am building the **Stayez Synthetic Data Studio**. As a premium Kenyan travel booking platform, Stayez wants to build a personalized AI recommendation engine. However, we need large volumes of realistic user behavior data to train and test the RAG/ML algorithms.

This tool uses Open-Source models (Llama 3.2 via Hugging Face, Llama 3.3 via Groq) and Gemini to generate highly realistic, structured synthetic data. 

### Features:
- Generates 3 core datasets: **Guest Profiles**, **Property Catalogs**, and **Booking Histories**.
- Outputs strict JSON data, instantly convertible to Pandas DataFrames and CSVs.
- Uses **Hugging Face InferenceClient** for open-source model execution.
- Interactive **Gradio UI** to configure and preview the synthetic data.
- Downloadable CSV outputs ready for AI fine-tuning or vector embedding.


In [ ]:
# Cell 1: Imports
import os
import json
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import InferenceClient

In [ ]:
# Cell 2: API Keys and Clients Setup
load_dotenv(override=True)

groq_api_key   = os.getenv('GROQ_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
hf_token       = os.getenv('HF_TOKEN')

# Hugging Face Inference Client (Week 3 Core Skill)
hf_client = InferenceClient(token=hf_token)

# Groq Client (Fastest for generation)
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# Gemini Client (Smart fallback)
gemini = OpenAI(api_key=gemini_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

MODEL_HF     = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_GROQ   = "llama-3.3-70b-versatile"
MODEL_GEMINI = "gemini-1.5-flash"

print("✅ API Clients initialized (Hugging Face, Groq, Gemini). Ready to generate data!")


In [ ]:
# Cell 3: Schemas for Recommendation System Data

DATASET_SCHEMAS = {
    "Guest Profiles": {
        "description": "Demographic & preference profiles for Stayez guests.",
        "fields": {
            "guest_id": "Unique string ID usually starting with GUEST_",
            "age": "Integer between 18 and 65",
            "travel_style": "String (e.g., Backpacker, Luxury, Business, Family, Solo)",
            "budget_per_night_ksh": "Integer (e.g., 2500 to 25000)",
            "preferred_activities": "List of strings (e.g., [Hiking, Nightlife, Beach, Quiet Retreat])"
        }
    },
    "Property Catalog": {
        "description": "A list of realistic Kenyan Airbnb-style properties.",
        "fields": {
            "property_id": "Unique string ID starting with PROP_",
            "name": "Catchy name for the property (e.g., 'Cozy Kilimani Studio')",
            "city": "String (e.g., Nairobi, Mombasa, Nakuru, Kisumu, Diani, Naivasha)",
            "type": "String (e.g., Villa, Studio, 1BR, 2BR, Cabin)",
            "price_ksh": "Integer (pricing matching the type)",
            "vibe_tags": "List of strings (e.g., [Romantic, Workspace, Scenic, Central])"
        }
    },
    "Booking & Review History": {
        "description": "Logs of past stays with ratings and text reviews. Needs to be realistic.",
        "fields": {
            "booking_id": "Unique string ID starting with BKG_",
            "guest_id": "String ID of the guest",
            "property_id": "String ID of the property",
            "nights_stayed": "Integer between 1 and 14",
            "rating": "Integer 1 to 5",
            "review_text": "A 1-2 sentence review justifying the rating. Make it sound like a real Kenyan or international guest."
        }
    }
}


In [ ]:
# Cell 4: The Generator Function

def generate_synthetic_data(dataset_type, num_records, model_choice):
    schema_info = DATASET_SCHEMAS[dataset_type]
    
    system_prompt = """
    You are a masterful synthetic data generation engine for Stayez, a Kenyan travel tech company.
    Your task is to generate highly realistic, varied, and structured mock data for our recommendation systems.
    You must return ONLY a raw JSON array format matching the requested schema.
    DO NOT wrap the JSON in markdown blocks like ```json ... ```. Just return the raw array [ { ... } ].
    DO NOT include any commentary.
    """
    
    user_prompt = f"""
    Generate exactly {num_records} records of type: {dataset_type}.
    Context: {schema_info['description']}
    
    The fields for each JSON object MUST be exactly:
    {json.dumps(schema_info['fields'], indent=2)}
    
    Make the data incredibly realistic to the Kenyan context (real city names, realistic KSh prices, relatable reviews).
    Output only the JSON array.
    """

    try:
        if model_choice == "Hugging Face (Llama 3.2 3B)":
            # Use Hugging Face Inference API
            response = hf_client.chat.completions.create(
                model=MODEL_HF,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.7,
                max_tokens=4000
            )
            raw_output = response.choices[0].message.content.strip()
            
        else:
            # Use Groq or Gemini
            client = groq if model_choice == "Groq (Llama 3 70B)" else gemini
            model = MODEL_GROQ if model_choice == "Groq (Llama 3 70B)" else MODEL_GEMINI
            
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.7,
                max_tokens=4000
            )
            raw_output = response.choices[0].message.content.strip()
        
        # Cleanup common LLM formatting artifacts just in case
        if raw_output.startswith("```json"):
            raw_output = raw_output[7:]
        if raw_output.startswith("```"):
            raw_output = raw_output[3:]
        if raw_output.endswith("```"):
            raw_output = raw_output[:-3]
            
        raw_output = raw_output.strip()
        
        # Parse into Python list of dictionaries
        data_list = json.loads(raw_output)
        
        # Convert to Pandas DataFrame for nice display and CSV export
        df = pd.DataFrame(data_list)
        
        # Save temporarily for Gradio to allow download
        csv_path = f"/tmp/stayez_{dataset_type.lower().replace(' ', '_')}.csv"
        df.to_csv(csv_path, index=False)
        
        return df, csv_path, "✅ Generation Successful!"
        
    except json.JSONDecodeError:
        return pd.DataFrame(), None, f" Error: The model did not return valid JSON.\n\nRaw output:\n{raw_output[:500]}"
    except Exception as e:
        return pd.DataFrame(), None, f"Unexpected Error: {str(e)}"


In [ ]:
# Cell 5: Gradio UI

with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as demo:
    gr.Markdown("## 🧪 Stayez Recommendation System - Synthetic Data Studio")
    gr.Markdown("Generate incredibly realistic, structured datasets using Llama 3 via Hugging Face/Groq to train Stayez ML models.")
    
    with gr.Row():
        with gr.Column(scale=1):
            dataset_dd = gr.Dropdown(
                choices=list(DATASET_SCHEMAS.keys()), 
                value="Guest Profiles", 
                label="Dataset Objective"
            )
            num_records_sl = gr.Slider(
                minimum=5, 
                maximum=50, 
                value=10, 
                step=5, 
                label="Number of Records"
            )
            model_dd = gr.Dropdown(
                choices=["Hugging Face (Llama 3.2 3B)", "Groq (Llama 3 70B)", "Gemini 1.5 Flash"], 
                value="Hugging Face (Llama 3.2 3B)", 
                label="Engine"
            )
            generate_btn = gr.Button("Generate Data ⚡", variant="primary")
            status_txt = gr.Textbox(label="Status", interactive=False)
            download_btn = gr.File(label="Download CSV", interactive=False)
            
        with gr.Column(scale=2):
            output_df = gr.Dataframe(label="Data Preview", wrap=True)

    generate_btn.click(
        fn=generate_synthetic_data,
        inputs=[dataset_dd, num_records_sl, model_dd],
        outputs=[output_df, download_btn, status_txt]
    )

demo.launch(inbrowser=True)
